In [0]:
%sql
CREATE OR REPLACE VIEW olist_catalog.olist_semantic.inventory_metrics
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: olist_catalog.olist_gold.fact_inventory
  comment: Inventory level metrics with stock status and product category analysis
  
  joins:
    - name: products
      source: olist_catalog.olist_gold.dim_products
      on: source.product_key = products.product_id
    - name: time_dim
      source: olist_catalog.olist_gold.dim_time
      on: source.snapshot_date_key = time_dim.date_key
  
  dimensions:
    - name: snapshot_date
      expr: time_dim.date
      display_name: Snapshot Date
      comment: Date when inventory snapshot was taken
      format:
        type: date
        date_format: year_month_day
      synonyms:
        - date
        - inventory date
    - name: snapshot_year_month
      expr: time_dim.year_month
      display_name: Snapshot Year-Month
      comment: Year and month of inventory snapshot
      synonyms:
        - month
        - period
    - name: product_category
      expr: source.product_category_name
      display_name: Product Category
      comment: Category of products in inventory
      synonyms:
        - category
        - product type
    - name: stock_status
      expr: source.stock_status
      display_name: Stock Status
      comment: Current stock status classification
      synonyms:
        - status
        - inventory status
    - name: weight_category
      expr: products.weight_category
      display_name: Weight Category
      comment: Product weight classification
      synonyms:
        - weight class
    - name: size_category
      expr: products.size_category
      display_name: Size Category
      comment: Product size classification
      synonyms:
        - size class
  
  measures:
    - name: total_stock_level
      expr: SUM(source.stock_level)
      display_name: Total Stock Level
      comment: Total inventory units across all products
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - stock
        - inventory level
        - units in stock
    - name: avg_stock_level
      expr: AVG(source.stock_level)
      display_name: Average Stock Level
      comment: Average inventory units per product
      format:
        type: number
        decimal_places:
          type: exact
          places: 1
      synonyms:
        - average stock
        - avg inventory
    - name: product_count
      expr: COUNT(DISTINCT source.product_key)
      display_name: Product Count
      comment: Number of unique products in inventory
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - products
        - SKU count
    - name: in_stock_product_count
      expr: SUM(source.in_stock_flag)
      display_name: In-Stock Product Count
      comment: Number of products currently in stock
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - available products
        - in stock count
    - name: stock_availability_rate
      expr: AVG(source.in_stock_flag)
      display_name: Stock Availability Rate
      comment: Percentage of products that are in stock
      format:
        type: percentage
        decimal_places:
          type: exact
          places: 1
      synonyms:
        - availability
        - in stock rate
    - name: max_stock_level
      expr: MAX(source.stock_level)
      display_name: Maximum Stock Level
      comment: Highest inventory level for any single product
      format:
        type: number
        decimal_places:
          type: exact
          places: 0
      synonyms:
        - max stock
        - peak inventory
$$